Imports

In [1]:
import pandas as pd
pop_df = pd.read_csv("policeforceareas2020to2024.csv")

In [2]:
pop_df.shape

(430, 175)

In [3]:
pop_df = pop_df.dropna(how="all")
pop_df.shape

(172, 175)

This code identifies all population columns that contain age‑ and gender‑specific counts (those starting with “F” or “M”) and removes any commas from their values. The raw population data stores numbers like "12,345" as strings, which prevents them from being treated as numeric values. Using .replace({",": ""}, regex=True) strips out the commas across all selected columns, allowing the data to be converted into integers later and used reliably for calculations such as totals, rates, or demographic summaries.

In [4]:
f_cols = [col for col in pop_df.columns if col.startswith("F")]
m_cols = [col for col in pop_df.columns if col.startswith("M")]
age_cols = f_cols + m_cols

# Remove commas
pop_df[age_cols] = pop_df[age_cols].replace({",": ""}, regex=True)

In [5]:
pop_df[age_cols] = pop_df[age_cols].astype(float)


In [6]:
pop_df[age_cols].dtypes

F0     float64
F1     float64
F2     float64
F3     float64
F4     float64
        ...   
M81    float64
M82    float64
M83    float64
M84    float64
M85    float64
Length: 172, dtype: object

In [7]:
# Rebuild column lists (just to be safe)
f_cols = [col for col in pop_df.columns if col.startswith("F")]
m_cols = [col for col in pop_df.columns if col.startswith("M")]

# Recalculate totals
pop_df["female_total"] = pop_df[f_cols].sum(axis=1)
pop_df["male_total"] = pop_df[m_cols].sum(axis=1)
pop_df["population_total"] = pop_df["female_total"] + pop_df["male_total"]

C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\1110874803.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df["female_total"] = pop_df[f_cols].sum(axis=1)
C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\1110874803.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df["male_total"] = pop_df[m_cols].sum(axis=1)
C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\1110874803.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

In [8]:
pop_df[["PFA 2023 Name", "Year", "female_total", "male_total", "population_total"]].head()

,PFA 2023 Name,Year,female_total,male_total,population_total
0,Metropolitan Police,2021.0,4532174.0,4263906.0,8796080.0
1,Metropolitan Police,2022.0,4572840.0,4286913.0,8859753.0
2,Metropolitan Police,2023.0,4634699.0,4351400.0,8986099.0
3,Metropolitan Police,2024.0,4673259.0,4401366.0,9074625.0
4,Cumbria,2021.0,254206.0,246523.0,500729.0


This line creates a lookup dictionary that maps each age‑specific population column to its corresponding numeric age. Since the population dataset stores ages in column names like "F30" or "M45", extracting the number part (30, 45, etc.) allows us to treat these columns as actual ages rather than labels. Building this age_lookup dictionary is the first step in calculating an average age for each area, because it provides the numeric age values needed to weight each population count correctly when computing a mean age.

In [9]:
f_cols = [col for col in pop_df.columns if col.startswith("F")]
m_cols = [col for col in pop_df.columns if col.startswith("M")]
age_cols = f_cols + m_cols

age_lookup = {col: int(col[1:]) for col in age_cols}

In [10]:
list(age_lookup.items())[:10]

[('F0', 0),
 ('F1', 1),
 ('F2', 2),
 ('F3', 3),
 ('F4', 4),
 ('F5', 5),
 ('F6', 6),
 ('F7', 7),
 ('F8', 8),
 ('F9', 9)]

creating a weighted pop column

In [11]:
for col in age_cols:
    pop_df[f"{col}_weighted"] = pop_df[col] * age_lookup[col]

C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\3431790232.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df[f"{col}_weighted"] = pop_df[col] * age_lookup[col]
C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\3431790232.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df[f"{col}_weighted"] = pop_df[col] * age_lookup[col]
C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\3431790232.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

In [12]:
pop_df.head(2)

,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M76_weighted,M77_weighted,M78_weighted,M79_weighted,M80_weighted,M81_weighted,M82_weighted,M83_weighted,M84_weighted,M85_weighted
0,E23000001,Metropolitan Police,2021.0,51645.0,52111.0,51165.0,51033.0,51843.0,52101.0,51137.0,...,1373700.0,1361129.0,1237392.0,1118324.0,1000880.0,1062558.0,1001548.0,936572.0,851424.0,4374100.0
1,E23000001,Metropolitan Police,2022.0,53387.0,51282.0,51393.0,50706.0,50616.0,51641.0,52319.0,...,1442404.0,1331484.0,1316094.0,1193927.0,1069840.0,951345.0,1004090.0,939643.0,877800.0,4485110.0


In [13]:
weighted_cols = [f"{col}_weighted" for col in age_cols]

pop_df["total_weighted_age"] = pop_df[weighted_cols].sum(axis=1)
pop_df[["PFA 2023 Name", "Year", "total_weighted_age"]].head()

C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\1179178840.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df["total_weighted_age"] = pop_df[weighted_cols].sum(axis=1)


,PFA 2023 Name,Year,total_weighted_age
0,Metropolitan Police,2021.0,324817840.0
1,Metropolitan Police,2022.0,327514063.0
2,Metropolitan Police,2023.0,332376988.0
3,Metropolitan Police,2024.0,336287866.0
4,Cumbria,2021.0,22380241.0


In [14]:
pop_df["average_age"] = pop_df["total_weighted_age"] / pop_df["population_total"]
pop_df[["PFA 2023 Name", "Year", "average_age"]].head()

C:\Users\roisi\AppData\Local\Temp\ipykernel_15868\1406920870.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop_df["average_age"] = pop_df["total_weighted_age"] / pop_df["population_total"]


,PFA 2023 Name,Year,average_age
0,Metropolitan Police,2021.0,36.927568
1,Metropolitan Police,2022.0,36.966500
2,Metropolitan Police,2023.0,36.987906
3,Metropolitan Police,2024.0,37.058045
4,Cumbria,2021.0,44.695316


In [15]:
pop_df["Year"] = pop_df["Year"].astype(int)
pop_df[["PFA 2023 Name", "Year", "average_age"]].head()


,PFA 2023 Name,Year,average_age
0,Metropolitan Police,2021,36.927568
1,Metropolitan Police,2022,36.966500
2,Metropolitan Police,2023,36.987906
3,Metropolitan Police,2024,37.058045
4,Cumbria,2021,44.695316


verification steps 

In [16]:
pop_df[age_cols].isna().sum().sum()

np.int64(0)

In [17]:
pop_df[age_cols].dtypes.unique()

array([dtype('float64')], dtype=object)

In [18]:
(
    (pop_df[f_cols].sum(axis=1) == pop_df["female_total"]).all(),
    (pop_df[m_cols].sum(axis=1) == pop_df["male_total"]).all(),
    ((pop_df[f_cols].sum(axis=1) + pop_df[m_cols].sum(axis=1)) == pop_df["population_total"]).all()
)

(np.True_, np.True_, np.True_)

In [19]:
row = 0
col = "F20"

pop_df.loc[row, f"{col}_weighted"], pop_df.loc[row, col] * 20

(np.float64(956100.0), np.float64(956100.0))

In [20]:
pop_df["average_age"].describe()

count    172.000000
mean      41.607598
std        2.038563
min       36.927568
25%       40.275537
50%       41.782491
75%       43.249765
max       45.421301
Name: average_age, dtype: float64

In [21]:
pop_df.duplicated().sum()

np.int64(0)

In [22]:
pop_df["Year"].unique()

array([2021, 2022, 2023, 2024])

exporting

In [23]:
pop_df.to_csv("population_clean.csv", index=False)